<a href="https://colab.research.google.com/github/org-mobicycle-productions/training/blob/main/training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LoRA Training — mobicycle.productionsTrains all LoRA adapters for the Productions account. One adapter per classification task.| Task | Labels | Status ||------|--------|--------|| relevance | not-relevant, actionable, contextual, evidential | 813 examples || priority | high, medium, low | Pending data || complexity | high, medium, low | Pending data || urgency | urgent, soon, normal, archive | Pending data |Base model: Mistral 7B Instruct v0.2.**Before running:** Upload `train.csv` for each task one at a time:1. Upload the task's `train.csv` to `/content/` via right-click → Upload to Colab2. Click "Run All" — the notebook finds it, trains, and outputs the adapter3. Download the adapter files from `/content/` via right-click → Download4. Repeat for the next task

In [17]:
# ── Config ──
ACCOUNT = "mobicycle.productions"
TASKS = ["relevance", "priority", "complexity", "urgency"]
MODEL = "mistralai/Mistral-7B-Instruct-v0.2"
MODEL_TYPE = "mistral"
EPOCHS = 3
BATCH_SIZE = 4
LEARNING_RATE = 2e-4
BLOCK_SIZE = 512
LORA_RANK = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
GRADIENT_ACCUMULATION = 4
# Persist config to disk (survives kernel restart)
import json
_cfg = {k: v for k, v in locals().items() if k.isupper()}
with open("/content/config.json", "w") as f:
    json.dump(_cfg, f)
print(f"Account: {ACCOUNT}")
print(f"Tasks: {', '.join(TASKS)}")
print(f"Model: {MODEL}")
print("Config saved to /content/config.json")

Account: mobicycle.productions
Tasks: relevance, priority, complexity, urgency
Model: mistralai/Mistral-7B-Instruct-v0.2
Config saved to /content/config.json


In [14]:
# 1. Install dependencies
!pip install -q torch transformers accelerate peft bitsandbytes datasets trl

In [15]:
# 2. Verify GPU
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    mem = getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)
    print(f"VRAM: {mem / 1e9:.1f} GB")
else:
    raise RuntimeError("No GPU! Select Kernel > Colab > connect to a GPU runtime.")

CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


In [18]:
# 3. Find training data + balance distribution
import os, csv, shutil, json, random

# Reload config if kernel restarted
if 'ACCOUNT' not in dir():
    try:
        with open('/content/config.json') as f:
            globals().update(json.load(f))
        print("Config reloaded from disk")
    except FileNotFoundError:
        raise RuntimeError("Run cell 1 (Config) first!")

active_tasks = []

for task in TASKS:
    data_dir = f"/content/data/{task}"
    os.makedirs(data_dir, exist_ok=True)
    train_path = f"{data_dir}/train.csv"

    if os.path.exists(train_path):
        print(f"[{task}] train.csv found")
    else:
        task_specific = f"/content/{task}-train.csv"
        generic = "/content/train.csv"
        if os.path.exists(task_specific):
            shutil.move(task_specific, train_path)
        elif os.path.exists(generic):
            shutil.move(generic, train_path)
        else:
            print(f"[{task}] No data — upload {task}-train.csv to /content/ first")
            continue

    # Parse rows by label
    rows_by_label = {}
    with open(train_path) as f:
        reader = csv.reader(f)
        header = next(reader)
        for row in reader:
            text = row[0] if row else ""
            if "[/INST]" in text:
                label = text.split("[/INST]")[-1].replace("</s>", "").strip()
                rows_by_label.setdefault(label, []).append(row)

    total = sum(len(v) for v in rows_by_label.values())
    if total < 10:
        print(f"[{task}] Only {total} examples — skipping")
        continue

    print(f"\n[{task}] ORIGINAL: {total} examples")
    for label, rows in sorted(rows_by_label.items()):
        print(f"  {label}: {len(rows)} ({len(rows)/total*100:.1f}%)")

    # Balance: oversample minority classes to match majority
    max_count = max(len(v) for v in rows_by_label.values())
    balanced_rows = []
    for label, rows in rows_by_label.items():
        if len(rows) < max_count:
            oversampled = rows * (max_count // len(rows))
            oversampled += random.sample(rows, max_count % len(rows))
            balanced_rows.extend(oversampled)
        else:
            balanced_rows.extend(rows)
    random.shuffle(balanced_rows)

    with open(train_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(header)
        writer.writerows(balanced_rows)

    print(f"  BALANCED: {len(balanced_rows)} examples ({max_count} per class)")
    active_tasks.append(task)

with open('/content/active_tasks.json', 'w') as f:
    json.dump(active_tasks, f)

print(f"\n{'='*50}")
print(f"Will train: {', '.join(active_tasks) if active_tasks else 'NONE'}")
print(f"{'='*50}")


[relevance] ORIGINAL: 813 examples
  not-relevant: 11 (1.4%)
  relevant-actionable: 765 (94.1%)
  relevant-contextual: 29 (3.6%)
  relevant-evidential: 8 (1.0%)
  BALANCED: 3060 examples (765 per class)
[priority] No data — upload priority-train.csv to /content/ first
[complexity] No data — upload complexity-train.csv to /content/ first
[urgency] No data — upload urgency-train.csv to /content/ first

Will train: relevance


In [19]:
# 4. Train each task (direct transformers + peft — 4-bit quantized)
import sys, types, json

# Reload state if kernel restarted
if 'ACCOUNT' not in dir():
    try:
        with open('/content/config.json') as f:
            globals().update(json.load(f))
        print("Config reloaded from disk")
    except FileNotFoundError:
        raise RuntimeError("Run cell 1 (Config) first!")
if 'active_tasks' not in dir():
    with open('/content/active_tasks.json') as f:
        active_tasks = json.load(f)
    print(f"Active tasks reloaded: {active_tasks}")

# Fix Colab's broken triton — create missing triton.ops module
try:
    import triton
    if not hasattr(triton, 'ops'):
        triton.ops = types.ModuleType('triton.ops')
        sys.modules['triton.ops'] = triton.ops
except ImportError:
    pass

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import load_dataset
import torch, os, math

trained_tasks = []

for task in active_tasks:
    project = f"{task}-{ACCOUNT}"
    data_dir = f"/content/data/{task}"
    output_dir = f"/content/{project}"

    print(f"\n{'='*50}")
    print(f"Training: {project}")
    print(f"{'='*50}\n")

    # Load dataset
    ds = load_dataset("csv", data_files=f"{data_dir}/train.csv", split="train")
    print(f"Loaded {len(ds)} examples")

    # Calculate warmup steps (10% of total steps)
    steps_per_epoch = math.ceil(len(ds) / (BATCH_SIZE * GRADIENT_ACCUMULATION))
    total_steps = steps_per_epoch * EPOCHS
    warmup_steps = int(total_steps * 0.1)

    # Load model in 4-bit with fp16 compute
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.model_max_length = BLOCK_SIZE
    model = AutoModelForCausalLM.from_pretrained(
        MODEL, quantization_config=bnb_config, device_map={"": 0},
        dtype=torch.float16, trust_remote_code=True
    )

    # Cast non-quantized layers (norms, embeddings) for stable training
    model = prepare_model_for_kbit_training(model)
    print(f"Model loaded: {MODEL} (4-bit, fp16)")

    # LoRA config
    lora_config = LoraConfig(
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    )

    # Training args — max_grad_norm=0 disables grad clipping (avoids GradScaler bf16 bug)
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION,
        learning_rate=LEARNING_RATE,
        gradient_checkpointing=True,
        max_grad_norm=0.0,
        warmup_steps=warmup_steps,
        weight_decay=0.01,
        logging_steps=10,
        save_strategy="epoch",
        report_to="none",
    )

    # Train — dataset "text" column is auto-detected
    trainer = SFTTrainer(
        model=model,
        train_dataset=ds,
        args=training_args,
        peft_config=lora_config,
        processing_class=tokenizer,
    )

    print("Starting training...")
    trainer.train()

    # Save adapter only
    trainer.model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)

    trained_tasks.append(task)
    print(f"\nDone: {project}")

    # Free memory for next task
    del model, trainer, tokenizer
    torch.cuda.empty_cache()

# Persist trained_tasks to disk
with open('/content/trained_tasks.json', 'w') as f:
    json.dump(trained_tasks, f)

print(f"\n{'='*50}")
print(f"Trained: {', '.join(trained_tasks) if trained_tasks else 'NONE'}")
print(f"{'='*50}")


Training: relevance-mobicycle.productions



Generating train split: 0 examples [00:00, ? examples/s]

Loaded 3060 examples


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

...

Truncating train dataset:   0%|          | 0/3060 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Starting training...


Step,Training Loss
10,5.019661
20,3.456948
30,1.991300
40,1.308365
50,1.112943
60,0.885774
70,0.667834
80,0.546866
90,1.590714
100,0.606046



Done: relevance-mobicycle.productions

Trained: relevance


In [20]:
# 5. Patch for Workers AI + verify
import json, os

# Reload state if kernel restarted
if 'ACCOUNT' not in dir():
    try:
        with open('/content/config.json') as f:
            globals().update(json.load(f))
    except FileNotFoundError:
        raise RuntimeError("Run cell 1 (Config) first!")
if 'trained_tasks' not in dir():
    with open('/content/trained_tasks.json') as f:
        trained_tasks = json.load(f)

for task in trained_tasks:
    project = f"{task}-{ACCOUNT}"
    output_dir = f"/content/{project}"
    config_path = os.path.join(output_dir, "adapter_config.json")

    print(f"\n── {task} ──")

    if not os.path.exists(config_path):
        print(f"  MISSING: {config_path}")
        continue

    with open(config_path) as f:
        config = json.load(f)

    config["model_type"] = MODEL_TYPE
    with open(config_path, "w") as f:
        json.dump(config, f, indent=2)

    for name in ["adapter_model.safetensors", "adapter_config.json"]:
        path = os.path.join(output_dir, name)
        if os.path.exists(path):
            print(f"  {name}: {os.path.getsize(path) / 1e6:.1f} MB")
        else:
            print(f"  {name}: MISSING!")

    print(f"  r={config.get('r')}, alpha={config.get('lora_alpha')}, model_type={config.get('model_type')}")


── relevance ──
  adapter_model.safetensors: 13.7 MB
  adapter_config.json: 0.0 MB
  r=8, alpha=16, model_type=mistral


In [21]:
# 6. Copy adapters to /content/ for download
import shutil, os, json

# Reload state if kernel restarted
if 'ACCOUNT' not in dir():
    try:
        with open('/content/config.json') as f:
            globals().update(json.load(f))
    except FileNotFoundError:
        raise RuntimeError("Run cell 1 (Config) first!")
if 'trained_tasks' not in dir():
    with open('/content/trained_tasks.json') as f:
        trained_tasks = json.load(f)

for task in trained_tasks:
    project = f"{task}-{ACCOUNT}"
    output_dir = f"/content/{project}"

    print(f"\n── {task} ──")
    for name in ["adapter_model.safetensors", "adapter_config.json"]:
        src = os.path.join(output_dir, name)
        dst = f"/content/{task}-{name}"
        if os.path.exists(src):
            shutil.copy2(src, dst)
            size = os.path.getsize(dst) / 1e6
            print(f"  {dst} ({size:.1f} MB)")
        else:
            print(f"  MISSING: {src}")

    print(f"  Save to: models/fine-tuning/PEFT/LoRA/{task}/{ACCOUNT}/adapter/")

print(f"\nFiles copied to /content/. Right-click → Download from Colab in VS Code.")
print(f"Then upload to Workers AI: bun run upload-adapter.ts --account {ACCOUNT}")


── relevance ──
  /content/relevance-adapter_model.safetensors (13.7 MB)
  /content/relevance-adapter_config.json (0.0 MB)
  Save to: models/fine-tuning/PEFT/LoRA/relevance/mobicycle.productions/adapter/

Files copied to /content/. Right-click → Download from Colab in VS Code.
Then upload to Workers AI: bun run upload-adapter.ts --account mobicycle.productions
